# Preparación de datos

## 1. Biomasa viable medida de LAB004–009

Una sola figura con una matriz de 2 × 3 paneles, uno por experimento, con escalas compartidas para facilitar la comparación. Cada marcador corresponde a una muestra real y los segmentos unen las muestras disponibles en orden temporal. Se conserva la biomasa viable empleada por el modelo químico, en g/L (numéricamente equivalente a kg/m³), obtenida del recuento Oculyze con la conversión histórica de 30 pg/célula.

Se lee la tabla normalizada de origen, sin depender de resultados de otros notebooks. No se ajustan parámetros, suavizan datos ni crean muestras interpoladas. El eje temporal conserva la referencia histórica de cada experimento. Las salidas quedan integradas en el EXECUTED, sin exportar CSV ni archivos de figuras.
La línea vertical discontinua identifica el **segundo pulso nutricional según el calendario operativo** (`Fernanda Folch.ics`), convertido al mismo reloj histórico de las muestras. El horario del calendario es una referencia documental y no certifica por sí solo la ejecución. No se reemplaza por el cruce estimado de densidad 1040 ni por la fila `pulso_nut` del ajuste químico, que usa otros horarios. La tabla de horarios se muestra debajo para mantener esa diferencia visible.

### Exclusiones de biomasa confirmadas

Se excluyen de la tabla de biomasa preparada y de esta figura seis mediciones señaladas por el usuario: LAB004-10, LAB005-10 y LAB006-09 (115 h), y las últimas de LAB007/008 (L-13) y LAB009 (L-12), a las 160 h. Son exclusiones por revisión de calidad, no una detección estadística automática. La identificación usa LAB + ID + tiempo para evitar confundir números de muestra.

Los datos Oculyze de esas seis muestras se eliminaron del Excel fuente (concentraciones viable y total, y gemación) y sus concentraciones/biomasas derivadas se vaciaron en ambos CSV normalizados. Se conservan las filas y todas las demás mediciones, incluido el peso seco independiente. Las correcciones manuales de la muestra 4 de LAB004, LAB005 y LAB006 permanecen aplicadas. La tabla inferior documenta los identificadores eliminados y sus motivos, sin conservar sus valores antiguos. `biomass` contiene las 70 mediciones disponibles de la fuente ya depurada; no aplica una máscara adicional ni interpola puntos. Esta depuración no reejecuta las calibraciones ni otros notebooks.


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython

get_ipython().run_line_magic('matplotlib', 'inline')
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'fermentation_model/shared').is_dir())
DATA_PATH = ROOT / 'fermentation_model/shared/results/secondary_metabolite_data_review/secondary_metabolite_long.csv'
input_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
LABS = tuple(f'LAB{i:03d}' for i in range(4, 10))
BIOMASS_COLUMN = 'X_viable_kg_m3'

raw = pd.read_csv(DATA_PATH)
biomass = raw.loc[raw.medium.eq('natural') & raw.batch.isin(LABS),
                  ['batch', 'sample_id', 'time_h', BIOMASS_COLUMN]].copy()
for column in ['time_h', BIOMASS_COLUMN]:
    biomass[column] = pd.to_numeric(biomass[column], errors='coerce')
# Trazabilidad de la depuración ya aplicada a las fuentes; no es un filtro del notebook.
exclusion_specs = (
    ('LAB004', 'ING26-LAB004-10', 115.0, 'Caída aislada con recuperación posterior'),
    ('LAB005', 'ING26-LAB005-10', 115.0, 'Caída aislada con recuperación posterior'),
    ('LAB006', 'ING26-LAB006-09', 115.0, 'Caída aislada con recuperación posterior'),
    ('LAB007', 'ING26-LAB007-L-13', 160.0, 'Última muestra: caída abrupta; exclusión confirmada'),
    ('LAB008', 'ING26-LAB008-L-13', 160.0, 'Última muestra: caída abrupta; exclusión confirmada'),
    ('LAB009', 'ING26-LAB009-L-12', 160.0, 'Última muestra: caída abrupta; exclusión confirmada'),
)
biomass_fields = ['X_viable_mcells_ml', 'X_total_mcells_ml', 'X_dead_direct_mcells_ml',
                 'X_dead_mcells_ml', 'X_viable_kg_m3', 'X_total_kg_m3', 'X_dead_kg_m3']
removal_rows = []
for lab, sample_id, time_h, reason in exclusion_specs:
    match = (raw.medium.eq('natural') & raw.batch.eq(lab) & raw.sample_id.eq(sample_id)
             & raw.time_h.sub(time_h).abs().lt(1e-6))
    assert match.sum() == 1, (lab, sample_id, time_h)
    assert raw.loc[match, biomass_fields].isna().all().all()
    removal_rows.append({'batch':lab, 'sample_id':sample_id, 'time_h':time_h,
                         'estado':'Oculyze eliminado de la fuente', 'motivo':reason})
removed_oculyze = pd.DataFrame(removal_rows)
biomass = biomass.dropna(subset=['time_h', BIOMASS_COLUMN]).sort_values(['batch', 'time_h'])
assert len(removed_oculyze) == 6 and len(biomass) == 70
# Las tres correcciones manuales siguen presentes.
for lab, expected in [('LAB004', 1.044), ('LAB005', 1.5576), ('LAB006', 0.9195)]:
    point = biomass.loc[biomass.batch.eq(lab) & biomass.sample_id.eq(f'ING26-{lab}-04')]
    assert len(point) == 1 and abs(point.iloc[0][BIOMASS_COLUMN]-expected) < 1e-12
print('Mediciones Oculyze eliminadas de la fuente (otras variables conservadas):')
display(removed_oculyze)
assert tuple(sorted(biomass.batch.unique())) == LABS
assert not biomass.duplicated(['batch', 'time_h']).any()
print('Fuente:', DATA_PATH.relative_to(ROOT))
print('Muestras por LAB:', biomass.groupby('batch').size().to_dict())
# Referencia temporal: el mismo t0 archivado para estas series históricas.
import re
CALENDAR_PATH = ROOT / 'fermentation_model/data/Laboratorio 2026/raw_data/Fernanda Folch.ics'
CLOCK_PATH = ROOT / 'fermentation_model/laboratory_2026/results/estimability_historical_natural/process_temperature_30min.csv'
nutrition_input_hashes = {p:hashlib.sha256(p.read_bytes()).hexdigest() for p in (CALENDAR_PATH,CLOCK_PATH)}
clocks = pd.read_csv(CLOCK_PATH,usecols=['batch','t0']).drop_duplicates()
clocks = clocks.loc[clocks.batch.isin(LABS)].set_index('batch')
assert clocks.index.is_unique and set(clocks.index)==set(LABS)
calendar_text = re.sub(r'\r?\n[ \t]','',CALENDAR_PATH.read_text(encoding='utf-8-sig'))
nutrition_rows = []
for event in calendar_text.split('BEGIN:VEVENT')[1:]:
    event = event.split('END:VEVENT')[0]
    fields = dict(line.split(':',1) for line in event.splitlines() if ':' in line)
    if 'Pulso nutricional 2' not in fields.get('SUMMARY',''):
        continue
    match = re.search(r'LAB00[4-9]',event)
    if match is None:
        continue
    lab = match.group()
    event_utc = pd.to_datetime(fields['DTSTART'],format='%Y%m%dT%H%M%SZ',utc=True)
    t0_local = pd.Timestamp(clocks.loc[lab,'t0']).tz_localize('America/Santiago')
    nutrition_rows.append({'LAB':lab,'hora_calendario_local':event_utc.tz_convert('America/Santiago').isoformat(),
        't_nutricion_h':(event_utc-t0_local).total_seconds()/3600,
        'fuente':'Calendario operativo · pulso 2'})
nutrition = pd.DataFrame(nutrition_rows).sort_values('LAB').reset_index(drop=True)
assert tuple(nutrition.LAB)==LABS
nutrition_times = nutrition.set_index('LAB').t_nutricion_h.to_dict()
display(nutrition)

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True,
                         layout='constrained')
colors = plt.get_cmap('tab10').colors
for ax, lab, color in zip(axes.flat, LABS, colors):
    samples = biomass.loc[biomass.batch.eq(lab)]
    ax.plot(samples.time_h, samples[BIOMASS_COLUMN], color=color,
            marker='o', markersize=5, linewidth=1.7)
    t_nut = nutrition_times[lab]
    ax.axvline(t_nut, color='#444444', linestyle='--', linewidth=1.4, zorder=3)
    ax.text(t_nut + 3, 0.96, f'Nutrición 2: {t_nut:g} h\n(calendario)',
            transform=ax.get_xaxis_transform(), va='top', ha='left', fontsize=8,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=2))
    ax.set_title(lab)
    ax.margins(x=0.02)
    ax.grid(alpha=0.3)
    ax.tick_params(labelbottom=True, labelleft=True)
axes.flat[0].set_ylim(0, float(biomass[BIOMASS_COLUMN].max()) * 1.08)
fig.suptitle('Biomasa viable medida · LAB004–009 · exclusiones de calidad aplicadas', fontsize=14)
fig.supxlabel('Tiempo histórico [h]')
fig.supylabel('Biomasa viable [g/L]')
plt.show()
plt.close(fig)
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == input_hash
assert all(hashlib.sha256(p.read_bytes()).hexdigest()==h for p,h in nutrition_input_hashes.items())